# Notebook 01: Hello, Redis!

Welcome to your Redis learning journey! In this first notebook, we'll understand what Redis is, why it's so popular, and write our very first Redis commands.

**Prerequisites:** Python 3, a running Redis server (we have one via Docker on port 6379).

---
## What is Redis?

**Redis** stands for **RE**mote **DI**ctionary **S**erver.

Think of it like this:

- You know **Python dictionaries**? `{"name": "Sujit", "age": 25}` — you store data as key-value pairs and look things up instantly.
- **Redis is exactly that**, but:
  - It lives **outside** your program, so **any** application can access it
  - It stores everything in **RAM** (memory), so it's **blazingly fast** — we're talking 100,000+ operations per second
  - It can **persist** data to disk so you don't lose it on restart
  - It supports **more than just strings** — lists, sets, sorted sets, hashes, streams, and more

### Real-World Analogy

| | Traditional Database (MySQL, PostgreSQL) | Redis |
|---|---|---|
| **Like...** | A filing cabinet | A sticky-note board |
| **Where data lives** | On disk (hard drive) | In memory (RAM) |
| **Speed** | Milliseconds | Microseconds |
| **Structure** | Tables, rows, columns | Key-value pairs |
| **Best for** | Complex queries, relationships | Speed, caching, real-time data |

---
## Why Do People Use Redis?

Redis is used by **Twitter, GitHub, StackOverflow, Instagram, Amazon** and thousands of other companies. Here's why:

1. **Caching** — Store frequently accessed data (like user profiles) to avoid hitting a slow database every time
2. **Session Storage** — Store web session data (logged-in users) with auto-expiry
3. **Message Queues** — Pass messages between different parts of your application
4. **Leaderboards** — Sorted sets make ranking systems trivial
5. **Rate Limiting** — Track how many API calls a user has made
6. **Real-time Analytics** — Count page views, clicks, etc. in real-time
7. **Pub/Sub** — Real-time notifications and chat systems

---
## How Redis Works

Redis uses a **client-server model**:

```
┌──────────────┐         TCP (port 6379)        ┌──────────────┐
│  Your Python  │  ──── SET name "Sujit" ────►  │              │
│  Program      │                                │  Redis       │
│  (Client)     │  ◄──────── OK ──────────────  │  Server      │
│              │                                │  (in RAM)    │
│              │  ──── GET name ────────────►  │              │
│              │  ◄──────── "Sujit" ──────────  │  ┌────────┐  │
└──────────────┘                                │  │ name   │  │
                                                │  │= Sujit │  │
┌──────────────┐                                │  └────────┘  │
│  Another App  │  ──── GET name ────────────►  │              │
│  (Client 2)   │  ◄──────── "Sujit" ──────────  │              │
└──────────────┘                                └──────────────┘
```

- The **server** holds all data in memory
- **Clients** connect via TCP on port **6379** (default)
- Multiple clients can connect simultaneously
- Commands are **single-threaded** — one at a time, no race conditions

---
## Let's Start! Step 1: Verify Setup

In [ ]:
# Install the redis Python package (skip if already installed)
!pip install redis

In [ ]:
# First, let's make sure the redis Python package is installed
import redis
print(f"redis-py version: {redis.__version__}")
print("Redis package is ready!")

## Step 2: Connect to Redis

We connect using the `redis.Redis()` class. Let's understand each parameter:

- `host='localhost'` — Redis server address (it's on our machine)
- `port=6379` — Default Redis port
- `db=0` — Redis has 16 databases (0-15), we'll use the default one
- `decode_responses=True` — **Important!** Without this, Redis returns bytes (`b'hello'`). With it, we get clean strings (`'hello'`)

In [ ]:
# Connect to Redis
r = redis.Redis(
    host='localhost',
    port=6379,
    db=0,
    decode_responses=True  # So we get strings, not bytes
)

# Test the connection with PING — Redis should reply PONG
response = r.ping()
print(f"Redis PING response: {response}")  # True means PONG received
print("Connected to Redis successfully!")

If you see `True` above, you're connected! If you get a `ConnectionError`, make sure your Redis Docker container is running:
```bash
docker start redis-learn
```

---
## Step 3: Your First Redis Command — SET and GET

This is the **"Hello World"** of Redis. We'll store a value and retrieve it.

- **SET** stores a key-value pair: `SET key value`
- **GET** retrieves the value: `GET key`

In [ ]:
# Redis CLI equivalent: SET greeting "Hello, Redis!"
result = r.set('greeting', 'Hello, Redis!')
print(f"SET result: {result}")  # True = success

# Redis CLI equivalent: GET greeting
value = r.get('greeting')
print(f"GET result: {value}")  # 'Hello, Redis!'

**What just happened?**

1. We told Redis: *"Store the value `'Hello, Redis!'` under the key `'greeting'`"*
2. Redis saved it in memory and said `True` (OK)
3. We asked: *"What's the value stored under key `'greeting'`?"*
4. Redis replied: `'Hello, Redis!'`

That's it — you've just used Redis! Everything else builds on this simple concept.

In [ ]:
# Let's try a few more SET/GET examples
r.set('name', 'Sujit')
r.set('language', 'Python')
r.set('learning', 'Redis')

print(f"Name: {r.get('name')}")
print(f"Language: {r.get('language')}")
print(f"Learning: {r.get('learning')}")

In [ ]:
# What happens if we GET a key that doesn't exist?
result = r.get('nonexistent_key')
print(f"Non-existent key returns: {result}")  # None
print(f"Type: {type(result)}")  # <class 'NoneType'>

---
## Step 4: Understanding Keys and Values

### Keys
- Keys are **always strings**
- They can be any string (even an empty one, but don't do that!)
- Maximum key size: **512 MB** (but keep them short!)

### Key Naming Convention
Use **colons (:)** as separators to create a namespace structure:

```
user:1001:name      → "Sujit"
user:1001:email     → "sujit@example.com"
session:abc123      → "session_data_here"
cache:homepage      → "<html>...</html>"
product:42:price    → "999"
```

This makes keys **organized** and **searchable** (you can find all `user:1001:*` keys).

In [ ]:
# Good key naming in practice
r.set('user:1001:name', 'Sujit')
r.set('user:1001:email', 'sujit@example.com')
r.set('user:1001:city', 'Mumbai')

r.set('user:1002:name', 'Alice')
r.set('user:1002:email', 'alice@example.com')

# Now we can easily get all info about user 1001
print(f"Name:  {r.get('user:1001:name')}")
print(f"Email: {r.get('user:1001:email')}")
print(f"City:  {r.get('user:1001:city')}")

---
## Step 5: More Essential Commands

### Overwriting a Key
SET always overwrites the previous value — no questions asked.

In [ ]:
r.set('color', 'blue')
print(f"Before: {r.get('color')}")

r.set('color', 'red')  # Overwrites!
print(f"After:  {r.get('color')}")

### INCR / DECR — Atomic Counters

Redis can treat string values as numbers and increment/decrement them **atomically** (safely, even with multiple clients).

In [ ]:
# Redis CLI: SET counter 0
r.set('page_views', 0)
print(f"Starting value: {r.get('page_views')}")

# Redis CLI: INCR counter
r.incr('page_views')       # +1 → 1
r.incr('page_views')       # +1 → 2
r.incr('page_views')       # +1 → 3
print(f"After 3 increments: {r.get('page_views')}")

# Redis CLI: INCRBY counter 10
r.incrby('page_views', 10) # +10 → 13
print(f"After +10: {r.get('page_views')}")

# Redis CLI: DECR counter
r.decr('page_views')       # -1 → 12
print(f"After decrement: {r.get('page_views')}")

**Why is this special?** Imagine 100 users viewing a page at the same time. With a regular database, you'd need locks to safely increment. With Redis, `INCR` is **atomic** — it's always safe, even with concurrent access.

### MSET / MGET — Set or Get Multiple Keys at Once

In [ ]:
# Redis CLI: MSET name "Sujit" lang "Python" topic "Redis"
r.mset({
    'profile:name': 'Sujit',
    'profile:language': 'Python',
    'profile:topic': 'Redis'
})
print("Set 3 keys at once!")

# Redis CLI: MGET name lang topic
values = r.mget('profile:name', 'profile:language', 'profile:topic')
print(f"Got 3 values at once: {values}")

### EXISTS — Check if a Key Exists

In [ ]:
# Redis CLI: EXISTS name
print(f"'profile:name' exists? {r.exists('profile:name')}")    # 1 (true)
print(f"'ghost_key' exists?    {r.exists('ghost_key')}")        # 0 (false)

# You can check multiple keys at once — returns count of existing keys
count = r.exists('profile:name', 'ghost_key', 'profile:language')
print(f"2 out of 3 keys exist: {count}")

### DEL — Delete Keys

In [ ]:
# Redis CLI: DEL color
r.set('temp_key', 'I will be deleted')
print(f"Before delete: {r.get('temp_key')}")

deleted_count = r.delete('temp_key')
print(f"Deleted {deleted_count} key(s)")
print(f"After delete: {r.get('temp_key')}")  # None

### KEYS — List All Keys (Use with Caution!)

In [ ]:
# Redis CLI: KEYS *
all_keys = r.keys('*')
print(f"All keys in the database ({len(all_keys)} total):")
for key in sorted(all_keys):
    print(f"  {key} = {r.get(key)}")

print("\n--- Only user keys ---")
# Redis CLI: KEYS user:*
user_keys = r.keys('user:*')
for key in sorted(user_keys):
    print(f"  {key} = {r.get(key)}")

> **WARNING:** Never use `KEYS *` in production! It scans ALL keys and blocks the server. For large databases, use `SCAN` instead (covered in Notebook 06).

---
## Step 6: The `db` Parameter — Multiple Databases

Redis has **16 databases** by default, numbered 0 to 15. They're completely isolated — keys in db 0 are invisible to db 1.

Think of them like 16 separate sticky-note boards in the same room.

In [ ]:
# Connect to database 0 (default)
r0 = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

# Connect to database 1
r1 = redis.Redis(host='localhost', port=6379, db=1, decode_responses=True)

# Set the same key in both databases
r0.set('message', 'I am in database 0')
r1.set('message', 'I am in database 1')

print(f"DB 0: {r0.get('message')}")
print(f"DB 1: {r1.get('message')}")

# Clean up db 1
r1.flushdb()

---
## Step 7: Useful Info Commands

In [ ]:
# How many keys are in the current database?
# Redis CLI: DBSIZE
print(f"Number of keys: {r.dbsize()}")

# What type is a key?
# Redis CLI: TYPE greeting
print(f"Type of 'greeting': {r.type('greeting')}")

# Server info (just a few fields)
info = r.info('server')
print(f"Redis version: {info['redis_version']}")
print(f"TCP port: {info['tcp_port']}")

# Memory info
mem = r.info('memory')
print(f"Used memory: {mem['used_memory_human']}")

---
## Step 8: Cleanup

Two ways to clear data:
- `FLUSHDB` — deletes all keys in the **current** database
- `FLUSHALL` — deletes all keys in **all 16** databases (nuclear option!)

In [ ]:
print(f"Keys before cleanup: {r.dbsize()}")

# Redis CLI: FLUSHDB
r.flushdb()

print(f"Keys after cleanup: {r.dbsize()}")  # 0

---
## Key Takeaways

| What You Learned | Details |
|---|---|
| **Redis** is an in-memory key-value store | Super fast, like a shared dictionary |
| **SET / GET** | Store and retrieve values |
| **INCR / DECR** | Atomic counters |
| **MSET / MGET** | Bulk set/get |
| **EXISTS / DEL** | Check existence, delete keys |
| **KEYS** | List keys (careful in production!) |
| **Key naming** | Use colons: `user:1001:name` |
| **decode_responses=True** | Always use this to get strings instead of bytes |

### Redis Command Cheat Sheet So Far
```
SET key value          → Store a value
GET key                → Retrieve a value
DEL key                → Delete a key
EXISTS key             → Check if key exists (1/0)
INCR key               → Increment by 1
INCRBY key amount      → Increment by N
DECR key               → Decrement by 1
MSET k1 v1 k2 v2      → Set multiple keys
MGET k1 k2             → Get multiple keys
KEYS pattern           → Find keys matching pattern
DBSIZE                 → Count of keys
FLUSHDB                → Delete all keys in current DB
PING                   → Test connection
```

---
## Exercises

Try these on your own! Add code cells below.

1. **Store your info:** SET your name, favorite food, and favorite movie in Redis using good key naming (e.g., `me:name`, `me:food`, `me:movie`). Then GET and print all three.

2. **Like counter:** Create a key `post:101:likes` starting at 0. Increment it 5 times using `INCR`. Print the final count.

3. **Bulk operations:** Use `MSET` to store 5 country capitals (e.g., `capital:india` → `New Delhi`). Use `MGET` to retrieve them all at once.

4. **Key detective:** After exercises 1-3, use `KEYS` to find: (a) all keys starting with `me:`, (b) all keys starting with `capital:`, (c) the total number of keys with `DBSIZE`.

In [ ]:
# Exercise 1: Your turn! Write your code here


In [ ]:
# Exercise 2: Like counter


In [ ]:
# Exercise 3: Bulk operations


In [ ]:
# Exercise 4: Key detective


---
**Next up: [Notebook 02 — Strings](./02_Strings.ipynb)** — We'll dive deep into Redis strings, counters, expiry, and build a simple rate limiter!